In [1]:
!pip install pygithub langchain langchain-community openai tiktoken pinecone-client langchain_pinecone sentence-transformers groq --quiet


[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [1]:
import os
import tempfile
from typing import Dict, List, Optional, Any
from pathlib import Path

from dotenv import load_dotenv
from github import Github, Repository
from git import Repo
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

from langchain.agents import Tool, load_tools, initialize_agent, AgentType
from langchain.memory import ConversationBufferMemory
from langchain.llms import OpenAI
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain.schema import Document

from langchain.embeddings import OpenAIEmbeddings
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore

from pinecone import Pinecone

In [2]:
"""
CodebaseRAG Tool - Integration of codebase_rag.py with ResearchAgent
"""

class CodebaseRAGTool:
    """
    A wrapper class that adapts the codebase_rag.py functionality to work as a tool within ResearchAgent.
    """
    
    def __init__(self, pinecone_api_key=None, index_name="codebase-rag"):
        """
        Initialize the CodebaseRAG tool.
        
        Args:
            pinecone_api_key: API key for Pinecone vector database
            index_name: Name of the Pinecone index to use
        """
        self.pinecone_api_key = pinecone_api_key
        self.index_name = index_name
        self.pinecone_index = None
        self.generator = None
        self.repositories = {}  # Track repositories that have been indexed
        
        # Supported file extensions and ignored directories
        self.SUPPORTED_EXTENSIONS = {'.py', '.js', '.tsx', '.jsx', '.ipynb', '.java',
                                    '.cpp', '.ts', '.go', '.rs', '.vue', '.swift', '.c', '.h'}
        self.IGNORED_DIRS = {'node_modules', 'venv', 'env', 'dist', 'build', '.git',
                            '__pycache__', '.next', '.vscode', 'vendor'}
        
        # Initialize dependencies if API key is provided
        if pinecone_api_key:
            self.initialize()
    
    def initialize(self):
        """Initialize Pinecone connection and text generation pipeline"""
        try:
            # Set up Pinecone
            os.environ['PINECONE_API_KEY'] = self.pinecone_api_key
            self.pinecone_index = self.connect_to_vector_store_index(self.index_name)
            
            # Set up text generation pipeline
            self.generator = self.create_text_generation_pipeline("Qwen/Qwen2.5-Coder-3B-Instruct")
            return True
        except Exception as e:
            print(f"Error initializing CodebaseRAG tool: {str(e)}")
            return False
    
    def connect_to_vector_store_index(self, index_name):
        """
        Connect to Pinecone vector store index.
        
        Args:
            index_name: Name of the Pinecone index
            
        Returns:
            Pinecone index object
        """
        # Initialize Pinecone
        pc = Pinecone(api_key=self.pinecone_api_key)
        
        # Connect to your Pinecone index
        pinecone_index = pc.Index(index_name)
        return pinecone_index
    
    def clone_repository(self, repo_url):
        """
        Clones a GitHub repository to a temporary directory.
        
        Args:
            repo_url: The URL of the GitHub repository.
            
        Returns:
            The path to the cloned repository.
        """
        repo_name = repo_url.split("/")[-1]  # Extract repository name from URL
        repo_path = tempfile.mkdtemp()
        repo_full_path = os.path.join(repo_path, repo_name)
        
        try:
            Repo.clone_from(repo_url, repo_full_path)
            return repo_full_path
        except Exception as e:
            print(f"Error cloning repository: {str(e)}")
            if os.path.exists(repo_full_path):
                return repo_full_path
            return None
    
    def get_file_content(self, file_path, repo_path):
        """
        Get content of a single file.
        
        Args:
            file_path: Path to the file
            repo_path: Root path of the repository
            
        Returns:
            Dictionary with file name and content
        """
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read()
            # Get relative path from repo root
            rel_path = os.path.relpath(file_path, repo_path)
            return {
                "name": rel_path,
                "content": content
            }
        except Exception as e:
            print(f"Error processing file {file_path}: {str(e)}")
            return None
    
    def get_main_files_content(self, repo_path):
        """
        Get content of supported code files from the local repository.
        
        Args:
            repo_path: Path to the local repository
            
        Returns:
            List of dictionaries containing file names and contents
        """
        files_content = []
        try:
            for root, _, files in os.walk(repo_path):
                # Skip if current directory is in ignored directories
                if any(ignored_dir in root for ignored_dir in self.IGNORED_DIRS):
                    continue
                # Process each file in current directory
                for file in files:
                    file_path = os.path.join(root, file)
                    if os.path.splitext(file)[1] in self.SUPPORTED_EXTENSIONS:
                        file_content = self.get_file_content(file_path, repo_path)
                        if file_content:
                            files_content.append(file_content)
        except Exception as e:
            print(f"Error reading repository: {str(e)}")
        return files_content
    
    def get_huggingface_embeddings(self, text, model_name="sentence-transformers/all-mpnet-base-v2"):
        """
        Generate embeddings using HuggingFace models.
        
        Args:
            text: Text to embed
            model_name: Name of the model to use
            
        Returns:
            Embeddings for the text
        """
        model = SentenceTransformer(model_name)
        return model.encode(text)
    
    def create_vector_store(self, repo_url):
        """
        Create a vector store from a repository.
        
        Args:
            repo_url: URL of the repository
            
        Returns:
            PineconeVectorStore object
        """
        path = self.clone_repository(repo_url)
        if not path:
            raise Exception(f"Failed to clone repository: {repo_url}")
            
        file_content = self.get_main_files_content(path)
        documents = []
        for file in file_content:
            doc = Document(
                page_content=f"{file['name']}\n{file['content']}",
                metadata={"source": file['name'], "text": f"{file['name']}\n{file['content']}"}
            )
            documents.append(doc)
        
        vectorstore = PineconeVectorStore.from_documents(
            documents=documents,
            embedding=HuggingFaceEmbeddings(),
            index_name=self.index_name,
            namespace=repo_url
        )
        return vectorstore
    
    def create_augmented_query(self, query, repo_url):
        """
        Create an augmented query with context from the vector store.
        
        Args:
            query: User query
            repo_url: Repository URL to search in
            
        Returns:
            Augmented query with context
        """
        raw_query_embedding = self.get_huggingface_embeddings(query)
        top_matches = self.pinecone_index.query(
            vector=raw_query_embedding.tolist(), 
            top_k=3,  # Retrieve more context
            include_metadata=True, 
            namespace=repo_url
        )
        
        contexts = [item['metadata']['text'] for item in top_matches['matches']]
        augmented_query = "<CONTEXT>\n" + "\n\n-------\n\n".join(contexts[:3]) + "\n-------\n</CONTEXT>\n\n\n\nMY QUESTION:\n" + query
        return augmented_query
    
    def create_text_generation_pipeline(self, model_id):
        """
        Create a text generation pipeline using HuggingFace models.
        
        Args:
            model_id: ID of the model to use
            
        Returns:
            Text generation pipeline
        """
        # Load tokenizer and model
        tokenizer = AutoTokenizer.from_pretrained(model_id)
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",  # Automatically selects GPU if available
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
        )
        
        # Create a pipeline
        generator = pipeline("text-generation", model=model, tokenizer=tokenizer)
        return generator
    
    def generate_response(self, query, repo_url):
        """
        Generate a response to a query using the text generation pipeline.
        
        Args:
            query: User query
            repo_url: Repository URL to search in
            
        Returns:
            Generated response
        """
        augmented_query = self.create_augmented_query(query, repo_url)
        
        # Define prompts
        system_prompt = (
            """You are a Senior Software Engineer specializing in code analysis.

        Analyze the provided code context carefully, considering:
        1. The structure and relationships between code components
        2. The specific implementation details and patterns
        3. The filepath and location of each code segment
        4. The type of code segment (e.g., function, class, etc.)
        5. The name of the code segment

        When answering questions:
        - Reference specific parts of the code and their locations
        - Explain the reasoning behind the implementation
        - Suggest improvements if relevant to the question
        - Consider the broader context of the codebase
        - Always use the code context to answer the question
        - Take a step by step approach in your problem-solving
        """
        )
        
        prompt = f"<s>[INST] {system_prompt}\n\n{augmented_query} [/INST]"
        result = self.generator(prompt, max_new_tokens=512, pad_token_id=self.generator.tokenizer.eos_token_id)
        return result[0]["generated_text"]
    
    def index_repository(self, repo_url):
        """
        Index a repository in the vector store.
        
        Args:
            repo_url: URL of the repository
            
        Returns:
            Status message
        """
        if repo_url in self.repositories:
            return f"Repository {repo_url} is already indexed."
        
        try:
            self.create_vector_store(repo_url)
            self.repositories[repo_url] = True
            return f"Successfully indexed repository: {repo_url}"
        except Exception as e:
            return f"Error indexing repository: {str(e)}"
    
    def search_codebase(self, query, repo_url=None):
        """
        Search the codebase and return relevant code snippets.
        
        Args:
            query: User query
            repo_url: Repository URL to search in (optional)
            
        Returns:
            Search results or error message
        """
        if not self.pinecone_index or not self.generator:
            return "CodebaseRAG tool is not initialized. Please provide a Pinecone API key."
        
        if not repo_url and not self.repositories:
            return "No repositories indexed. Please index a repository first."
        
        # If repo_url is not specified but we have repositories indexed, use the first one
        if not repo_url:
            repo_url = next(iter(self.repositories))
        
        try:
            response = self.generate_response(query, repo_url)
            # Extract the actual response part (after the instruction)
            response_parts = response.split("[/INST]")
            if len(response_parts) > 1:
                return response_parts[1].strip()
            return response
        except Exception as e:
            return f"Error searching codebase: {str(e)}"


In [3]:
class ResearchAgent:
    def __init__(self, llm):
        self.llm = llm
        self.memory = ConversationBufferMemory(memory_key="chat_history")
        
        self.codebase_rag_tool = None
       
            
        # Define tools
        self.tools = [
            Tool(
                name="Wikipedia",
                func=self.search_wikipedia,
                description="Useful for searching Wikipedia articles"
            ),
            Tool(
                name="Calculator",
                func=self.calculate,
                description="Useful for performing mathematical calculations"
            )
        ]
        pinecone_api_key = "pcsk_2tvYAf_CRu8eU6x71bqgt14giRmCfEojwh4yQSuiXYjn6FHdM1ro3ghrCEiPFEHDU3s7Y6"
        if pinecone_api_key:
            self.init_codebase_rag_tool(pinecone_api_key, "https://github.com/YousefAllam221b/ProgrammingPractice")
            
        # Initialize agent
        self.agent = initialize_agent(
            self.tools,
            self.llm,
            agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
            memory=self.memory,
            verbose=True
        )
        
    def init_codebase_rag_tool(self, pinecone_api_key, repo_url=None):
        """
        Initialize the CodebaseRAG tool with Pinecone API key and optional repository URL.
        
        Args:
            pinecone_api_key: API key for Pinecone vector database
            repo_url: URL of the repository to index (optional)
            
        Returns:
            Status message
        """
        self.codebase_rag_tool = CodebaseRAGTool(pinecone_api_key)
        
        # Add the tool to the agent's tools
        self.tools.append(
            Tool(
                name="CodebaseSearch",
                func=self.search_codebase,
                description="Useful for searching through codebases to find relevant code snippets and answer code-related questions"
            )
        )
        
        # Index repository if provided
        result = None
        if repo_url:
            result = self.codebase_rag_tool.index_repository(repo_url)
        
        # Reinitialize agent with updated tools
        if hasattr(self, 'agent'):
            self.agent = initialize_agent(
                self.tools,
                self.llm,
                agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
                memory=self.memory,
                verbose=True
            )
        
        return result if result else "CodebaseRAG tool initialized successfully."

    def search_codebase(self, query, repo_url=None):
        """
        Search the codebase using the CodebaseRAG tool.
        
        Args:
            query: User query
            repo_url: Repository URL to search in (optional)
            
        Returns:
            Search results or error message
        """
        if not self.codebase_rag_tool:
            return "CodebaseRAG tool is not initialized. Please provide a Pinecone API key."
        
        return self.codebase_rag_tool.search_codebase(query, repo_url)
    
    def search_wikipedia(self, query):
        from langchain.tools import WikipediaQueryRun
        wikipedia = WikipediaQueryRun(api_wrapper=Wikipedia())
        return wikipedia.run(query)
    
    def calculate(self, expression):
        return eval(expression)
    
    def run(self, query):
        return self.agent.run(query)


In [5]:
# from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# bnb_config = BitsAndBytesConfig(
#         load_in_4bit=True,
#         bnb_4bit_use_double_quant=True,
#         bnb_4bit_quant_type="nf4",
#         bnb_4bit_compute_dtype=torch.float16,  # Changed from bfloat16 to float16
#         bnb_4bit_quant_storage=torch.uint8,    # Added for storage optimization
#         use_nested_quant=True,                 # Added for nested quantization
#     )

# model = AutoModelForCausalLM.from_pretrained(
#     "Qwen/Qwen2.5-Coder-7B-Instruct", quantization_config=bnb_config
# )

# tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Coder-7B-Instruct")
# from langchain.llms import HuggingFacePipeline

# # Build pipeline
# hf_pipeline = pipeline(
#     "text-generation",
#     model=model,
#     tokenizer=tokenizer,
#     max_new_tokens=512,
#     temperature=0.7,
#     top_p=0.95,
#     repetition_penalty=1.1
# )

# # Wrap in LangChain-compatible format
# llm = HuggingFacePipeline(pipeline=hf_pipeline)

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_community.llms import HuggingFacePipeline
from langchain.agents import initialize_agent, AgentType
from langchain.tools import DuckDuckGoSearchRun
from transformers import BitsAndBytesConfig

# Load a model like Mistral
# model_name = "mistralai/Mistral-7B-Instruct-v0.2"
model_name = "Qwen/Qwen2.5-Coder-7B-Instruct"

# Define quantization config properly
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# Load model using quantization config
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map=0,  # Let accelerate place layers on GPU/CPU
    trust_remote_code=True

)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.7,
    do_sample=True,
    repetition_penalty=1.1,
    device=0
)

llm = HuggingFacePipeline(pipeline=pipe)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 34.00 MiB. GPU 

In [6]:
!nvidia-smi

Fri May 30 23:36:21 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A4500               Off |   00000000:65:00.0  On |                  Off |
| 30%   30C    P8             17W /  200W |   20134MiB /  20470MiB |      1%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [21]:
tokenizer = LlamaTokenizer.from_pretrained(
  'teknium/OpenHermes-2.5-Mistral-7B',
  trust_remote_code=True
)

model = MistralForCausalLM.from_pretrained(
    "teknium/OpenHermes-2.5-Mistral-7B",
    torch_dtype=torch.float16,
    device_map="auto", #{'': 'cuda:0'},
    load_in_8bit=False,
    load_in_4bit=True,
    use_flash_attention_2=False,
    low_cpu_mem_usage=True
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.7,
    do_sample=True,
    repetition_penalty=1.1
    pad_token_id = pipeline.tokenizer.eos_token_id
)

llm = HuggingFacePipeline(pipeline=pipe)

NameError: name 'LlamaTokenizer' is not defined

In [ ]:
generated_ids = model.generate(model_inputs, pad_token_id=tokenizer.pad_token_id, max_new_tokens=2000, do_sample=True)

In [20]:
 llm("Tell me a fun fact about space.")

KeyboardInterrupt: 

In [11]:
research_agent = ResearchAgent(llm)

/usr/local/lib/python3.10/dist-packages/accelerate/utils/modeling.py:1405: UserWarning: Current model requires 603984384 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/tmp/ipykernel_623/3363698211.py:178: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embedding=HuggingFaceEmbeddings(),
/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [12]:
research_agent.run('who is mark zuckerberg')



> Entering new AgentExecutor chain...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


KeyboardInterrupt: 